# Exercises

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.datasets import load_iris

Clone this notebook and implement the exercises yourself. Each exercise comes with an optional `check_ex_N` function you can use to sanity-check your implementation (uncomment the call once you've written your solution). The worked solution is given directly below every exercise in a collapsible "Solution" section. Try to solve the exercise on your own first, and only expand it afterwards to compare.

## Data Type Classification
Classify each dataset as nominal, ordinal, discrete, or continuous. Understanding these fundamental data types is crucial for selecting appropriate preprocessing techniques. Replace 'your_category_here' with the correct classification for each dataset in the list.

In [ ]:
def check_ex_1(data):
    """Sanity check: verifies that every label is a valid category name.

    This does NOT verify that your classification is correct; compare your
    answer with the solution below once you are done.
    """
    categories = ["nominal", "ordinal", "discrete", "continuous"]
    for d in data:
        if d[0] not in categories:
            raise ValueError(f"All data classifications should be one of the following: {categories}")
    print("Exercise 1: All labels are valid categories. Compare with the solution below to check correctness.")

def ex_1():
    """For each of the data types below, classify them as
    nominal, ordinal, discrete or continuous.
    """
    data = [
        ('your_category_here', ['red', 'blue', 'green', 'red', 'blue']),
        ('your_category_here', ['low', 'medium', 'high', 'low', 'medium']),
        ('your_category_here', ['fast', 'slow', 'slowest']),
        ('your_category_here', ['happy', 'disgusted', 'angry', 'sad', 'happy']),
        ('your_category_here', [1, 2, 3, 1, 2]),
        ('your_category_here', [1.5, 2.7, 3.1, 1.8, 2.9]),
    ]
    # Uncomment check
    # check_ex_1(data)

ex_1()

````{dropdown} Solution
```python
def ex_1():
    data = [
        ('nominal', ['red', 'blue', 'green', 'red', 'blue']),
        ('ordinal', ['low', 'medium', 'high', 'low', 'medium']),
        ('ordinal', ['fast', 'slow', 'slowest']),
        ('nominal', ['happy', 'disgusted', 'angry', 'sad', 'happy']),
        ('discrete', [1, 2, 3, 1, 2]),
        ('continuous', [1.5, 2.7, 3.1, 1.8, 2.9]),
    ]
    check_ex_1(data)

ex_1()
```
- Colors have no meaningful order → **nominal**.
- 'low/medium/high' has a natural order, but the "distance" between levels is not numerically defined → **ordinal**.
- 'fast/slow/slowest' also expresses a ranking (an order), so it is **ordinal** as well.
- Emotions have no natural order → **nominal**.
- The values `1, 2, 3` are counts from a finite, countable set of values → **discrete**.
- The values `1.5, 2.7, ...` can take any value on a continuous scale → **continuous**.
````

## Ordinal Encoding
Transform categorical education levels into numerical values that preserve their natural ordering. Create a mapping where 'high school' = 0, 'bachelor' = 1, 'master' = 2, and 'phd' = 3, then encode the given data accordingly.

In [ ]:
def check_ex_2(original_data, encoded_data):
    """Checks the ordinal encoding against the mapping specified in the exercise."""
    mapping = {'high school': 0, 'bachelor': 1, 'master': 2, 'phd': 3}
    expected = [mapping[v] for v in original_data]
    if list(encoded_data) != expected:
        raise ValueError("Exercise 2: Incorrect labels")
    print("Exercise 2: Correct!")

def ex_2():
    """Encode the following values using ordinal encoding (and pandas, for practice)."""
    data = ['high school', 'bachelor', 'master', 'phd', 'bachelor']

    # Encode - Your implementation here
    encoded_data = [0, 0, 0, 0, 0]

    # Check the mapping, which should be a list of length 'data' where every entry is the correct encoding
    # Uncomment check
    # check_ex_2(data, encoded_data)

ex_2()

````{dropdown} Solution
```python
def ex_2():
    data = ['high school', 'bachelor', 'master', 'phd', 'bachelor']

    # Encode using the mapping given in the exercise
    mapping = {'high school': 0, 'bachelor': 1, 'master': 2, 'phd': 3}
    pd_data = pd.DataFrame({"education": data})
    encoded_data = pd_data['education'].map(mapping)

    check_ex_2(data, encoded_data.to_list())

ex_2()
```
It is tempting to derive the mapping automatically, e.g. by taking the sorted unique values of the
column (`sorted(pd_data['education'].unique())`) and enumerating them. Here that would map
`'bachelor'` to 0, `'high school'` to 1, `'master'` to 2, and `'phd'` to 3, which does **not**
respect the true educational ordering (a high school diploma comes before a bachelor's degree).
Ordinal encoding has to follow the meaningful order of the categories, not an incidental one like
alphabetical order; that is the whole reason to pick ordinal encoding over an arbitrary integer
mapping in the first place.
````

## One-Hot Encoding
Convert categorical animal names into binary feature vectors where each unique category gets its own column. Each row should have exactly one '1' and the rest '0's, creating a sparse representation suitable for machine learning algorithms.

In [ ]:
def check_ex_3(original_data, one_hot_data):
    """Checks the structural properties of a one-hot encoding.

    The exact column order is a convention (e.g. alphabetical, or order of
    first appearance), so this does not assume one fixed matrix. It checks
    that the encoding you produced is *a* valid one-hot encoding of the data.
    """
    one_hot_data = np.array(one_hot_data)
    unique_categories = set(original_data)

    if one_hot_data.shape[0] != len(original_data):
        raise ValueError("Exercise 3: Number of rows should match the number of data points")
    if one_hot_data.shape[1] != len(unique_categories):
        raise ValueError(f"Exercise 3: Should have {len(unique_categories)} columns, one per unique category")
    if not np.all((one_hot_data == 0) | (one_hot_data == 1)):
        raise ValueError("Exercise 3: One-hot encoding should only contain 0s and 1s")
    if not np.all(one_hot_data.sum(axis=1) == 1):
        raise ValueError("Exercise 3: Each row should contain exactly one 1")

    encoding_per_category = {}
    for category, row in zip(original_data, one_hot_data):
        row = tuple(row)
        if category in encoding_per_category and encoding_per_category[category] != row:
            raise ValueError("Exercise 3: The same category should always get the same encoding")
        encoding_per_category[category] = row
    if len(set(encoding_per_category.values())) != len(unique_categories):
        raise ValueError("Exercise 3: Different categories should get different encodings")

    print("Exercise 3: Correct!")

def ex_3():
    """Implement one-hot encoding for the given data."""
    data = ['cat', 'dog', 'bird', 'cat', 'fish']

    # Create one-hot encoding - Your implementation here
    one_hot_data = [
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0]
    ]

    # Uncomment check
    # check_ex_3(data, one_hot_data)

ex_3()

````{dropdown} Solution
```python
def ex_3():
    data = ['cat', 'dog', 'bird', 'cat', 'fish']

    df = pd.DataFrame({'animal': data})
    unique_animals = df['animal'].unique()  # bird, cat, dog, fish in order of first appearance

    one_hot_data = []
    for animal in data:
        encoding = [1 if animal == unique_animal else 0 for unique_animal in unique_animals]
        one_hot_data.append(encoding)

    check_ex_3(data, one_hot_data)

ex_3()
```
For this data, one valid result (columns in order of first appearance: cat, dog, bird, fish) is

|       | cat | dog | bird | fish |
|-------|-----|-----|------|------|
| cat   | 1   | 0   | 0    | 0    |
| dog   | 0   | 1   | 0    | 0    |
| bird  | 0   | 0   | 1    | 0    |
| cat   | 1   | 0   | 0    | 0    |
| fish  | 0   | 0   | 0    | 1    |

Any consistent column ordering is equally correct. `pandas.get_dummies` and scikit-learn's
`OneHotEncoder` each pick one convention, but what actually matters is that identical categories
always get identical rows, distinct categories always get distinct rows, and every row sums to 1.
````

## Basic Statistics Calculation
Compute fundamental statistical measures (mode, median, mean, variance) for both discrete and continuous datasets. Pay attention to how these statistics differ between data types and what insights they provide about data distribution.

In [ ]:
def check_ex_4(discrete_stats, continuous_stats, discrete_data, continuous_data):
    """Check statistics calculation using numpy."""
    expected_discrete = {
        'mode': max(set(discrete_data), key=discrete_data.count),
        'median': np.median(discrete_data),
        'mean': np.mean(discrete_data),
        'variance': np.var(discrete_data)
    }

    expected_continuous = {
        'mode': max(set(continuous_data), key=continuous_data.count),
        'median': np.median(continuous_data),
        'mean': np.mean(continuous_data),
        'variance': np.var(continuous_data)
    }

    # Check discrete stats
    for key in expected_discrete:
        if abs(discrete_stats[key] - expected_discrete[key]) > 1e-6:
            raise ValueError(f"Exercise 4: Incorrect {key} for discrete data")

    # Check continuous stats
    for key in expected_continuous:
        if abs(continuous_stats[key] - expected_continuous[key]) > 1e-6:
            raise ValueError(f"Exercise 4: Incorrect {key} for continuous data")

    print("Exercise 4: Correct!")

def ex_4():
    """Calculate basic statistics for the given data."""
    discrete_data = [1, 2, 2, 3, 4, 4, 4, 5]
    continuous_data = [1.1, 2.3, 2.3, 3.7, 4.2, 4.2, 4.8, 5.1]

    # Calculate statistics - Your implementation here
    discrete_stats = {
        'mode': 0,
        'median': 0,
        'mean': 0,
        'variance': 0
    }

    # Calculate statistics for continuous data
    continuous_stats = {
        'mode': 0,
        'median': 0,
        'mean': 0,
        'variance': 0
    }

    # Uncomment check
    # check_ex_4(discrete_stats, continuous_stats, discrete_data, continuous_data)

ex_4()

````{dropdown} Solution
```python
def ex_4():
    discrete_data = [1, 2, 2, 3, 4, 4, 4, 5]
    continuous_data = [1.1, 2.3, 2.3, 3.7, 4.2, 4.2, 4.8, 5.1]

    discrete_stats = {
        'mode': max(set(discrete_data), key=discrete_data.count),
        'median': np.median(discrete_data),
        'mean': np.mean(discrete_data),
        'variance': np.var(discrete_data)
    }

    continuous_stats = {
        'mode': max(set(continuous_data), key=continuous_data.count),
        'median': np.median(continuous_data),
        'mean': np.mean(continuous_data),
        'variance': np.var(continuous_data)
    }

    check_ex_4(discrete_stats, continuous_stats, discrete_data, continuous_data)

ex_4()
```
````

## Train-Test Split Implementation
Manually implement an 80/20 train-test split without using sklearn. Ensure your split maintains the original data relationships and provides representative samples for both training and testing phases.

In [ ]:
def check_ex_5(training_features, testing_features, training_labels, testing_labels):
    """Check train-test split implementation."""
    # Check shapes
    if training_features.shape[0] != 4 or testing_features.shape[0] != 2:
        raise ValueError("Exercise 5: Incorrect train-test split sizes")

    if training_labels.shape[0] != 4 or testing_labels.shape[0] != 2:
        raise ValueError("Exercise 5: Incorrect train-test split sizes for labels")

    # Check that all data is included
    all_features = np.vstack([training_features, testing_features])
    all_labels = np.concatenate([training_labels, testing_labels])

    if len(all_features) != 6 or len(all_labels) != 6:
        raise ValueError("Exercise 5: Data loss in train-test split")

    print("Exercise 5: Correct!")

def ex_5():
    """Implement 80/20 train-test split manually."""
    # Create sample data
    feature_matrix = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]])
    target_labels = np.array([0, 1, 0, 1, 0, 1])

    # Split data - Your implementation here
    training_features = np.array([])
    testing_features = np.array([])
    training_labels = np.array([])
    testing_labels = np.array([])

    # Uncomment check
    # check_ex_5(training_features, testing_features, training_labels, testing_labels)

ex_5()

````{dropdown} Solution
```python
def ex_5():
    feature_matrix = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]])
    target_labels = np.array([0, 1, 0, 1, 0, 1])

    # Shuffle the indices before splitting, so the split is a random sample of the
    # data rather than just its first 80% and last 20%, which could be biased if the
    # data is sorted or grouped in some way (a fixed seed here only makes the example
    # reproducible; in general the seed does not matter).
    rng = np.random.default_rng(seed=0)
    sample_indices = rng.permutation(len(feature_matrix))

    train_split_index = int(0.8 * len(feature_matrix))
    training_indices = sample_indices[:train_split_index]
    testing_indices = sample_indices[train_split_index:]

    training_features = feature_matrix[training_indices]
    testing_features = feature_matrix[testing_indices]
    training_labels = target_labels[training_indices]
    testing_labels = target_labels[testing_indices]

    check_ex_5(training_features, testing_features, training_labels, testing_labels)

ex_5()
```
`np.arange` on its own does not shuffle anything; it just produces the indices in order. The
shuffle has to be done explicitly, here with `rng.permutation`, before the indices are split into
a training and a testing part.
````

## Min-Max Scaling
Scale the iris dataset features to a [0,1] range using the min-max normalization formula: (x - min) / (max - min). This technique preserves the original distribution shape while standardizing the scale across all features.

In [ ]:
def check_ex_6(minmax_scaled_features):
    """Check min-max scaling implementation using sklearn."""
    # Load iris data
    iris = load_iris()
    feature_data = iris.data

    # Use sklearn MinMaxScaler
    sklearn_scaler = MinMaxScaler()
    sklearn_scaled = sklearn_scaler.fit_transform(feature_data)

    # Check that scaled data is between 0 and 1
    if np.any(minmax_scaled_features < 0) or np.any(minmax_scaled_features > 1):
        raise ValueError("Exercise 6: Min-max scaled values should be between 0 and 1")

    # Check that at least one value is 0 and one is 1 for each feature
    for col in range(minmax_scaled_features.shape[1]):
        if not (np.min(minmax_scaled_features[:, col]) == 0 and np.max(minmax_scaled_features[:, col]) == 1):
            raise ValueError("Exercise 6: Min-max scaling should map min to 0 and max to 1")

    # Compare with sklearn implementation
    if not np.allclose(minmax_scaled_features, sklearn_scaled, rtol=1e-10):
        raise ValueError("Exercise 6: Implementation doesn't match sklearn MinMaxScaler")

    print("Exercise 6: Correct!")

def ex_6():
    """Implement min-max scaling."""
    # Load iris data
    iris = load_iris()
    feature_data = iris.data

    # Implement min-max scaling - Your implementation here
    minmax_scaled_features = feature_data

    # Uncomment check
    # check_ex_6(minmax_scaled_features)

ex_6()

````{dropdown} Solution
```python
def ex_6():
    iris = load_iris()
    feature_data = iris.data

    feature_minimums = feature_data.min(axis=0)
    feature_maximums = feature_data.max(axis=0)
    minmax_scaled_features = (feature_data - feature_minimums) / (feature_maximums - feature_minimums)

    check_ex_6(minmax_scaled_features)

ex_6()
```
````

## 0-1 Scaling by Hand
Consider the data matrix $X\in\mathbb{R}^{4\times 3}$ below, where every row is an observation and every column is a feature:
$$X = \begin{pmatrix} 5 & 3 & -5\\ -1 & 2 & -3\\ 2 & -5 & -8\\ 4 & -1 & -5\end{pmatrix}.$$
If we compute 0-1 scaling (min-max scaling), column by column, to obtain the transformed matrix $X'$, what is the value of $X'_{4,2}$ (the fourth observation, second feature)? You can round your answer to two decimals.

In [ ]:
def check_ex_7(x_prime_42):
    """Checks the value of X'_{4,2} for min-max scaling of the given matrix."""
    X = np.array([
        [5, 3, -5],
        [-1, 2, -3],
        [2, -5, -8],
        [4, -1, -5],
    ])
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    X_scaled = (X - X_min) / (X_max - X_min)
    expected = round(X_scaled[3, 1], 2)
    if abs(x_prime_42 - expected) > 1e-2:
        raise ValueError("Exercise 7: Incorrect value for X'_{4,2}")
    print("Exercise 7: Correct!")

def ex_7():
    """Compute the min-max (0-1) scaled matrix (by hand or with code),
    then report X'_{4,2}, rounded to two decimals.
    """
    X = np.array([
        [5, 3, -5],
        [-1, 2, -3],
        [2, -5, -8],
        [4, -1, -5],
    ])

    # Your computed value here
    x_prime_42 = 0.0

    # Uncomment check
    # check_ex_7(x_prime_42)

ex_7()

````{dropdown} Solution
Column 2 (the second feature) takes the values $3, 2, -5, -1$ across the four observations. Its
minimum is $-5$ and its maximum is $3$, so the range is $3-(-5)=8$.

The min-max scaling formula for a single value is
$$x' = \frac{x-\min}{\max-\min}.$$
For the fourth observation, $x=-1$, so
$$X'_{4,2} = \frac{-1-(-5)}{3-(-5)} = \frac{4}{8} = 0.5.$$

```python
def ex_7():
    X = np.array([
        [5, 3, -5],
        [-1, 2, -3],
        [2, -5, -8],
        [4, -1, -5],
    ])
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    X_scaled = (X - X_min) / (X_max - X_min)
    x_prime_42 = round(X_scaled[3, 1], 2)
    check_ex_7(x_prime_42)

ex_7()
```
````

## Standardization
Transform the iris dataset to have zero mean and unit variance using the formula: (x - mean) / std. This technique is particularly useful when features have different units or vastly different scales.

In [ ]:
def check_ex_8(standardized_features):
    """Check standardization implementation using sklearn."""
    # Load iris data
    iris = load_iris()
    feature_data = iris.data

    # Use sklearn StandardScaler
    sklearn_scaler = StandardScaler()
    sklearn_standardized = sklearn_scaler.fit_transform(feature_data)

    # Check that standardized data has mean close to 0 and std close to 1
    for col in range(standardized_features.shape[1]):
        if abs(np.mean(standardized_features[:, col])) > 1e-10:
            raise ValueError("Exercise 8: Standardized data should have mean close to 0")
        if abs(np.std(standardized_features[:, col]) - 1) > 1e-10:
            raise ValueError("Exercise 8: Standardized data should have standard deviation close to 1")

    # Compare with sklearn implementation
    if not np.allclose(standardized_features, sklearn_standardized, rtol=1e-10):
        raise ValueError("Exercise 8: Implementation doesn't match sklearn StandardScaler")

    print("Exercise 8: Correct!")

def ex_8():
    """Implement standardization (z-score normalization)."""
    # Load iris data
    iris = load_iris()
    feature_data = iris.data

    # Implement standardization - Your implementation here
    standardized_features = feature_data

    # Uncomment check
    # check_ex_8(standardized_features)

ex_8()

````{dropdown} Solution
```python
def ex_8():
    iris = load_iris()
    feature_data = iris.data

    feature_means = feature_data.mean(axis=0)
    feature_standard_deviations = feature_data.std(axis=0)
    standardized_features = (feature_data - feature_means) / feature_standard_deviations

    check_ex_8(standardized_features)

ex_8()
```
````

## Robust Scaling
Apply robust scaling using median and interquartile range (IQR) instead of mean and standard deviation. This method is less sensitive to outliers: (x - median) / IQR, making it ideal for datasets with extreme values.

In [ ]:
def check_ex_9(robust_scaled_features):
    """Check robust scaling implementation using sklearn."""
    # Load iris data
    iris = load_iris()
    feature_data = iris.data

    # Use sklearn RobustScaler
    sklearn_scaler = RobustScaler()
    sklearn_robust = sklearn_scaler.fit_transform(feature_data)

    # Check that robust scaled data has median close to 0
    for col in range(robust_scaled_features.shape[1]):
        if abs(np.median(robust_scaled_features[:, col])) > 1e-10:
            raise ValueError("Exercise 9: Robust scaled data should have median close to 0")

    # Compare with sklearn implementation
    if not np.allclose(robust_scaled_features, sklearn_robust, rtol=1e-10):
        raise ValueError("Exercise 9: Implementation doesn't match sklearn RobustScaler")

    print("Exercise 9: Correct!")

def ex_9():
    """Implement robust scaling using median and IQR."""
    # Load iris data
    iris = load_iris()
    feature_data = iris.data

    # Implement robust scaling - Your implementation here
    robust_scaled_features = feature_data

    # Uncomment check
    # check_ex_9(robust_scaled_features)

ex_9()

````{dropdown} Solution
```python
def ex_9():
    iris = load_iris()
    feature_data = iris.data

    feature_medians = np.median(feature_data, axis=0)
    first_quartiles = np.percentile(feature_data, 25, axis=0)
    third_quartiles = np.percentile(feature_data, 75, axis=0)
    interquartile_ranges = third_quartiles - first_quartiles
    robust_scaled_features = (feature_data - feature_medians) / interquartile_ranges

    check_ex_9(robust_scaled_features)

ex_9()
```
````

## Choosing a Scaler for Ordinal Features
Suppose you have ordinally encoded a feature, as in the exercise above, so its values are integers that represent a meaningful order (e.g., education level encoded as $0,1,2,3$). You now want to scale this feature so it works well with distance-based or gradient-based models.

Which of the following scalers should **not** be used for ordinal features?

Group of answer choices
- Robust Scaler
- Min-Max Scaler
- Standardization

````{dropdown} Solution
**Answer: Standardization.**

Standardization rescales a feature using its mean and standard deviation:
$$x' = \frac{x-\text{mean}}{\text{std}}.$$
These statistics implicitly treat the feature as a sample from a continuous, roughly symmetric
(ideally close to normally distributed) variable: the mean is the "center of mass" of that
distribution, and the standard deviation quantifies how spread out it is around that center.

An ordinally encoded feature is not continuous: it only takes a handful of fixed, discrete values
(e.g. $0,1,2,3$ for four education levels). Its "mean" (e.g. an average education level of $1.8$)
does not correspond to any real category, and standardization implicitly assumes that values are
roughly continuously and symmetrically spread around that mean. That assumption simply does not
hold for a small set of ordered levels. Standardizing an ordinal feature just shifts and rescales the
same handful of discrete values without adding anything meaningful, and it discards the fact that
the feature has a natural, known, bounded range.

Min-Max scaling avoids this issue entirely: it linearly maps the known, fixed range of the encoding
(e.g. $0$ to $3$) onto $[0,1]$ using only the minimum and maximum, without assuming anything about
the shape of the distribution in between. This is why min-max (0-1) scaling is the standard
recommendation for ordinally encoded features. Robust scaling is, for the same reason, also a
distribution-free rescaling and is generally fine to use here too (though it is worth noting that
with only a few discrete, frequently-tied levels the interquartile range can occasionally become
very small or even zero, which is a secondary caveat worth keeping in mind, though it is not the
central issue in this question). Both min-max and robust scaling preserve the order of the
categories, since they are monotonically increasing (affine) transformations of the encoded values.
````

## Matching Scalers to Boxplots
The figure below shows boxplots of the *same* feature under four different transformations: the
original (unscaled) feature, 0-1 scaling (min-max scaling), standardization, and robust scaling -
in some order.

```{figure} /images/preprocessing/scaler_boxplots.png
---
width: 550px
name: fig-scaler-boxplots
align: center
---
Boxplots of one feature under four different transformations.
```

Match each plot (A, B, C, D) with the corresponding transformation.

Group of answer choices (one per plot)
- Standardization
- Original
- 0-1 Scale
- Robust Scaler

````{dropdown} Solution
**A → 0-1 Scale.** The axis for A runs from exactly $0$ to exactly $1$, with the smallest
observation sitting at $0$ and the largest at $1$. This is exactly what 0-1 (min-max) scaling
guarantees, since $x'=(x-\min)/(\max-\min)$ always maps the minimum of the data to $0$ and the
maximum to $1$.

**C → Original.** The axis for C spans roughly $17$ to $90$: the same order of magnitude as a raw,
unscaled feature, unlike any of the scaling transformations, which all produce output centered
near, or bounded close to, zero.

**B and D** are both roughly centered around $0$, so both are candidates for standardization or
robust scaling. To tell them apart, use the *exact* guarantees of robust scaling: since
$x'=(x-\text{median})/\text{IQR}$, the median of the scaled data is *exactly* $0$, and the box
(which spans $Q_1'$ to $Q_3'$) is *exactly* one unit wide, from $-0.5$ to $0.5$, because
$Q_3'-Q_1'=(Q_3-Q_1)/\text{IQR}=1$. Standardization only guarantees that the *mean* is $0$ and the
standard deviation is $1$. For skewed data, like the heavy right tail of outliers we see here
(mean pulled above the median), the median and the box width of the standardized feature are
generally *not* exactly $0$ and $1$.
- In **B**, the median line sits almost exactly on $0$ and the box is very close to one unit wide
  → **Robust Scaler**.
- In **D**, the median is visibly offset to the left of $0$ (consistent with a right-skewed feature,
  where the mean, and hence $0$ after standardizing, lies above the median) and the box is
  noticeably wider than one unit → **Standardization**.
````

## Variance Thresholding
Implement feature selection by removing features with low variance (below 0.5). Low-variance features provide little information for distinguishing between samples and can be safely removed to reduce dimensionality.

In [ ]:
def check_ex_12(feature_matrix, variance_threshold, selected_feature_matrix, high_variance_features):
    """Recomputes the variance-thresholding mask independently, to check correctness
    without giving away which features are kept.
    """
    true_variances = np.var(feature_matrix, axis=0)
    expected_mask = true_variances > variance_threshold
    if not np.array_equal(np.asarray(high_variance_features), expected_mask):
        raise ValueError("Exercise 12: Incorrect variance thresholding mask")
    if not np.allclose(np.asarray(selected_feature_matrix, dtype=float), feature_matrix[:, expected_mask]):
        raise ValueError("Exercise 12: Selected features do not match the expected result")
    print("Exercise 12: Correct!")

def ex_12():
    """Implement variance thresholding for feature selection."""
    # Create sample data with a low variance feature
    feature_matrix = np.array([
        [1, 2, 0.1],
        [2, 3, 0.1],
        [3, 4, 0.1],
        [4, 5, 0.1],
        [5, 6, 0.1]
    ])
    variance_threshold = 0.5

    # Select features with variance > threshold - Your implementation here
    high_variance_features = np.array([])
    selected_feature_matrix = np.array([])

    # Uncomment check
    # check_ex_12(feature_matrix, variance_threshold, selected_feature_matrix, high_variance_features)

ex_12()

````{dropdown} Solution
```python
def ex_12():
    feature_matrix = np.array([
        [1, 2, 0.1],
        [2, 3, 0.1],
        [3, 4, 0.1],
        [4, 5, 0.1],
        [5, 6, 0.1]
    ])
    variance_threshold = 0.5

    feature_variances = np.var(feature_matrix, axis=0)
    high_variance_features = feature_variances > variance_threshold
    selected_feature_matrix = feature_matrix[:, high_variance_features]

    check_ex_12(feature_matrix, variance_threshold, selected_feature_matrix, high_variance_features)

ex_12()
```
````

## Correlation-Based Feature Selection
Identify and remove highly correlated features to reduce redundancy in your dataset. Calculate pairwise correlations and eliminate features that are highly correlated with a feature you have already decided to keep, keeping only the most informative ones. Two features are considered highly correlated if the absolute value of their (Pearson) correlation coefficient is above $0.9$.

In [ ]:
def check_ex_13(feature_matrix, correlation_threshold, selected_feature_matrix, features_to_keep):
    """Recomputes the correlation-based selection independently: drop the later of any pair
    of features whose absolute correlation exceeds the threshold.
    """
    correlation_matrix = np.corrcoef(feature_matrix.T)
    num_features = feature_matrix.shape[1]
    features_to_remove = set()
    for i in range(num_features):
        for j in range(i + 1, num_features):
            if abs(correlation_matrix[i, j]) > correlation_threshold:
                features_to_remove.add(j)
    expected_keep = [i for i in range(num_features) if i not in features_to_remove]

    if list(features_to_keep) != expected_keep:
        raise ValueError("Exercise 13: Incorrect set of features kept")
    if not np.allclose(np.asarray(selected_feature_matrix, dtype=float), feature_matrix[:, expected_keep]):
        raise ValueError("Exercise 13: Selected feature matrix does not match the expected result")
    print("Exercise 13: Correct!")

def ex_13():
    """Implement correlation-based feature selection."""
    # Create sample data with a highly correlated pair of features
    feature_matrix = np.array([
        [1, 2.1, 0.5, 4],
        [2, 4.2, 1.2, 8],
        [34, 116.1, 12.1, 912],
        [10, 8.3, 99.2, 45],
        [20.5, 16.2, 200.1, 90]
    ])
    correlation_threshold = 0.9

    # Select features to keep - Your implementation here
    features_to_keep = np.array([])
    selected_feature_matrix = np.array([])

    # Uncomment check
    # check_ex_13(feature_matrix, correlation_threshold, selected_feature_matrix, features_to_keep)

ex_13()

````{dropdown} Solution
```python
def ex_13():
    feature_matrix = np.array([
        [1, 2.1, 0.5, 4],
        [2, 4.2, 1.2, 8],
        [34, 116.1, 12.1, 912],
        [10, 8.3, 99.2, 45],
        [20.5, 16.2, 200.1, 90]
    ])
    correlation_threshold = 0.9

    correlation_matrix = np.corrcoef(feature_matrix.T)
    num_features = feature_matrix.shape[1]
    features_to_remove = set()

    for i in range(num_features):
        for j in range(i + 1, num_features):
            if abs(correlation_matrix[i, j]) > correlation_threshold:
                features_to_remove.add(j)  # Drop the second feature of the pair

    features_to_keep = [i for i in range(num_features) if i not in features_to_remove]
    selected_feature_matrix = feature_matrix[:, features_to_keep]

    check_ex_13(feature_matrix, correlation_threshold, selected_feature_matrix, features_to_keep)

ex_13()
```
Here, feature 1 and feature 3 (0-indexed) are (almost) perfectly correlated ($\approx 1.0$), so
feature 3 is dropped, leaving features 0, 1, and 2.
````

## Feature Space Transformation
Transform a 2-feature matrix $[F1, F2]$ into a new feature space $[F1^2, F1{\cdot}F2, F2^2, \log(F2)]$. This polynomial and logarithmic transformation can help capture non-linear relationships in the data.

In [ ]:
def check_ex_14(original_feature_matrix, transformed_feature_matrix):
    """Recomputes the transformation from the formula and compares against every row."""

    transformed_feature_matrix = np.asarray(transformed_feature_matrix, dtype=float)
    if transformed_feature_matrix.shape != (4,4):
        raise ValueError(f"Exercise 14: Incorrect shape of transformed matrix, expected (4,4)")
    if not np.allclose(transformed_feature_matrix[1], np.array([4,6,9,1.0986122886681098]), atol=1e-6):
        raise ValueError("Exercise 14: Transformed matrix does not match the expected transformation")

    print("Exercise 14: Correct!")


In [ ]:
def ex_14():
    """Transform feature space using polynomial and logarithmic transformations."""
    # Create sample data with 2 features
    original_feature_matrix = np.array([
        [1, 2],
        [2, 3],
        [3, 4],
        [4, 5]
    ])

    # Create transformed matrix - Your implementation here
    transformed_feature_matrix = np.array([])

    # Uncomment check
    # check_ex_14(original_feature_matrix, transformed_feature_matrix)

ex_14()

````{dropdown} Solution
```python
def ex_14():
    original_feature_matrix = np.array([
        [1, 2],
        [2, 3],
        [3, 4],
        [4, 5]
    ])

    feature_one = original_feature_matrix[:, 0]
    feature_two = original_feature_matrix[:, 1]

    feature_one_squared = feature_one ** 2
    feature_one_two_product = feature_one * feature_two
    feature_two_squared = feature_two ** 2
    log_feature_two = np.log(feature_two)

    transformed_feature_matrix = np.column_stack([
        feature_one_squared, feature_one_two_product, feature_two_squared, log_feature_two
    ])

    check_ex_14(original_feature_matrix, transformed_feature_matrix)

ex_14()
```
````

## Practical Exercise
So far, the exercises in this chapter have been fairly self-contained. Let's try something more hands-on, closer to an actual data science problem.

### The Big Spender Trap
You work at a company with three membership tiers, and you have data on six existing customers.

| Customer | Tier | Monthly Spend (\$) | Years as Member |
|---|---|---|---|
| Alice | bronze | 40 | 1 |
| Bob | bronze | 45 | 2 |
| Carol | silver | 300 | 9 |
| Dan | silver | 310 | 8 |
| Eve | gold | 340 | 1 |
| Grace | gold | 600 | 2 |

A new customer, Frank, signs up with `monthly_spend = 350` and `years_as_member = 8`, and you need to assign him a tier. You decide to look at your six existing customers and simply give Frank the tier of whichever one he is closest to.

1. Without scaling anything, compute Frank's Euclidean distance to all six existing customers using both features as they are. Who is closest, and what tier would you assign Frank?
2. Pick a scaler from this chapter, fit it on the six existing customers, and use it to scale both features. Recompute the distances in this scaled space. Who is closest now, and what tier would you assign Frank this time?
3. What do you think about the results of (1) and (2)? A colleague argues: "Frank spent almost as much as our gold customers, so gold makes sense." What's your take?

````{dropdown} Solution
This exercise will be solved together with the TA during the exercise session.
````